In [2]:
import numpy as np
import time

from sklearn.datasets  import fetch_openml
from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import(
    accuracy_score,
    classification_report
)


In [4]:
fashion_mnist=fetch_openml('Fashion-MNIST',version=1,as_frame=False)

X=fashion_mnist.data
y=fashion_mnist.target.astype(int)

print(f"Total dataset size: {X.shape[0]} images, each with {X.shape[1]} pixels.")


Total dataset size: 70000 images, each with 784 pixels.


In [5]:
X_subset,_,y_subset,_ =train_test_split(
    X,y,
    train_size=12000,
    stratify=y,
    random_state=42
)

In [6]:
X_train,X_test,y_train,y_test=train_test_split(
    X_subset,y_subset,
    test_size=2000,
    stratify=y_subset,
    random_state=42
)

print(f"Training images: {X_train.shape[0]}")
print(f"testing images: {X_test.shape[0]}")

Training images: 10000
testing images: 2000


In [8]:
print(f"Before scaling -> Min: {X_train.min()}, Max: {X_train.max()}")

# Rescale to range [0.0, 1.0]
X_train = X_train / 255.0
X_test = X_test / 255.0

# Check values after scaling
print(f"After scaling -> Min: {X_train.min()}, Max: {X_train.max()}")


Before scaling -> Min: 0, Max: 255
After scaling -> Min: 0.0, Max: 1.0


In [17]:
k_values = [1, 3, 5, 7, 9, 15]

results = {}

# Print header
print(f"{'K':<8} | {'Accuracy (%)':<12} | {'Prediction Time (s)':<20}")
print("-" * 48)

for k in k_values:

    # Initialize KNN
    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric='euclidean',
        n_jobs=-1
    )

    # KNN stores the training data
    knn.fit(X_train, y_train)

    # Start prediction timer
    start_time = time.time()

    # Predict test images
    y_pred = knn.predict(X_test)

    # Calculate prediction time
    elapsed_time = time.time() - start_time

    # Calculate accuracy
    acc = accuracy_score(y_test, y_pred)

    # Store results
    results[k] = {
        'accuracy': acc,
        'prediction_time': elapsed_time
    }

    # Print result
    print(
        f"{k:<8} | "
        f"{acc * 100:<12.2f} | "
        f"{elapsed_time:<20.2f}"
    )


K        | Accuracy (%) | Prediction Time (s) 
------------------------------------------------
1        | 79.30        | 0.21                
3        | 80.90        | 0.20                
5        | 81.00        | 0.20                
7        | 81.10        | 0.20                
9        | 80.95        | 0.21                
15       | 80.25        | 0.21                


In [19]:
class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

# Find the K with highest accuracy
best_k = max(
    results,
    key=lambda k: results[k]['accuracy']
)

print(
    f"Best K is: {best_k} "
    f"with {results[best_k]['accuracy'] * 100:.2f}% accuracy\n"
)

# Create KNN model using the best K
best_knn = KNeighborsClassifier(
    n_neighbors=best_k,
    metric='euclidean',
    n_jobs=-1
)

# Fit the model
best_knn.fit(X_train, y_train)

# Predict using the best model
best_predictions = best_knn.predict(X_test)

# Classification report
print("Per-Class Classification Report:")

print(
    classification_report(
        y_test,
        best_predictions,
        target_names=class_names
    )
)


Best K is: 7 with 81.10% accuracy

Per-Class Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.73      0.83      0.78       200
     Trouser       0.97      0.94      0.96       200
    Pullover       0.67      0.73      0.70       200
       Dress       0.85      0.83      0.84       200
        Coat       0.74      0.69      0.72       200
      Sandal       1.00      0.76      0.86       200
       Shirt       0.58      0.55      0.56       200
     Sneaker       0.81      0.92      0.86       200
         Bag       0.97      0.94      0.95       200
  Ankle boot       0.85      0.94      0.89       200

    accuracy                           0.81      2000
   macro avg       0.82      0.81      0.81      2000
weighted avg       0.82      0.81      0.81      2000

